# 12 - LLM-as-a-Judge & Structural Rule Checks

**Objective**: Perform multi-dimensional quality assessment of generated hypotheses using:
1. **Qualitative Rubrics (LLM-as-a-Judge)**: 6 dimensions (Domain Relevance, Testability, Clarity, Specificity, Mechanistic Novelty, Directional Plausibility) scored 1–5.
2. **Deterministic Structural Rule Suite**: 9 deterministic syntactic and scientific rules ensuring falsifiability, variable specification, and bounded length.

**Inputs**:
- `outputs/generated_hypotheses.json`
- `outputs/candidate_research_gaps.csv`

**Outputs**:
- `outputs/judge_results.json`
- `outputs/rule_check_results.json`

**Production Code Reused**:
- `src/phase2/llm_judge.py`
- `src/phase2/rule_checks.py`


In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

project_root = Path().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.phase2.rule_checks import run_all_checks, CheckDetail
from src.phase2.llm_judge import LLMJudge

print("Judge and rule check engines loaded.")


## 1. Inspect LLM Judge Rubric Evaluation
We load qualitative evaluations from `outputs/judge_results.json`.


In [ ]:
with open("outputs/judge_results.json", "r", encoding="utf-8") as f:
    judge_data = json.load(f)

rubric_scores = {
    "Domain Relevance": [],
    "Testability": [],
    "Clarity": [],
    "Specificity": [],
    "Novelty": [],
    "Plausibility": []
}

for h_id, eval_obj in judge_data.items():
    rubrics = eval_obj.get("rubrics", {})
    if isinstance(rubrics, dict):
        for k, v in rubrics.items():
            clean_k = k.replace("_", " ").title()
            if clean_k in rubric_scores and isinstance(v, (int, float)):
                rubric_scores[clean_k].append(v)

print("Mean LLM Judge Rubric Ratings (Scale 1 - 5):")
for dim, scores in rubric_scores.items():
    if scores:
        print(f"  - {dim:<20}: {np.mean(scores):.2f} / 5.00 (± {np.std(scores):.2f})")


## 2. Inspect 9 Deterministic Structural Rule Checks
Every hypothesis is subjected to 9 non-negotiable structural rules:
1. Independent and Dependent Variables present
2. Explicit target population
3. Falsifiability
4. Causal language calibration
5. Evidence references present
6. Evidence pool coverage
7. Specificity keywords
8. Bounded word length (20-200 words)
9. Contradictory evidence acknowledgment


In [ ]:
with open("outputs/rule_check_results.json", "r", encoding="utf-8") as f:
    rule_data = json.load(f)

print(f"Total Rule-Checked Hypotheses: {len(rule_data)}")

sample_h = list(rule_data.keys())[0]
sample_res = rule_data[sample_h]

print(f"\nRule Check Breakdown for {sample_h}:")
print(f"  Passed: {sample_res.get('passed_count', 9)} / 9")
print(f"  Warnings: {sample_res.get('warning_count', 0)}")
print(f"  Failed: {sample_res.get('failed_count', 0)}")

for check in sample_res.get("checks", []):
    status_icon = "[OK]" if check.get("status") == "pass" else ("[WARN]" if check.get("status") == "warning" else "[FAIL]")
    print(f"  [{status_icon}] {check.get('check_name'):<32}: {check.get('status').upper()} - {check.get('message')}")
